**Programmer: python_scripts (Abhijith Warrier)**

**PYTHON SCRIPT TO *EXPLORE HOW NUMBERS AND TEXT BECOME BYTES AND BINARY DIGITS. 🐍🔢***

A number and a letter can produce identical bytes. How does software know what those bytes mean?

Bits describe binary states; representations give them meaning. In this lab, we will encode an integer, inspect individual bits, compare text encodings, and decode bytes back into values. We will distinguish a Unicode code point from its encoded bytes and see why choosing the right interpretation matters.

---

## 1. Requirements
Python 3.9+ and Jupyter. The examples use built-in types only; no additional packages or imports are needed.

---

## 2. Eight bits, one byte
For an unsigned byte, bit positions have weights from 128 down to 1. The pattern `01000001` selects 64 and 1. Eight bits allow 2⁸ = 256 patterns, representing unsigned values 0–255.

In [1]:
value = 65
bits = f"{value:08b}"  # Text display, padded to at least eight binary digits.
weights = [128, 64, 32, 16, 8, 4, 2, 1]
selected = [weight for bit, weight in zip(bits, weights) if bit == "1"]
print("Bits:", bits)
print("Selected weights:", selected)
print("Sum:", sum(selected))
assert bits == "01000001" and sum(selected) == value

Bits: 01000001
Selected weights: [64, 1]
Sum: 65


---

## 3. Choose an integer representation
Serialize 65 as one unsigned byte. `to_bytes` creates bytes; the format string only displays them as binary text. This is not the internal memory layout or object size of a Python integer. A value that does not fit the chosen width raises an error.

In [2]:
number_bytes = value.to_bytes(1, byteorder="big", signed=False)
print("Hex:", number_bytes.hex(" "))
print("Bits:", " ".join(f"{byte:08b}" for byte in number_bytes))
print("Recovered integer:", int.from_bytes(number_bytes, "big", signed=False))
assert number_bytes == bytes([65])
assert int.from_bytes(number_bytes, "big") == value
try:
    (256).to_bytes(1, "big")
except OverflowError:
    print("256 cannot fit in one unsigned byte.")

Hex: 41
Bits: 01000001
Recovered integer: 65
256 cannot fit in one unsigned byte.


---

## 4. The same byte, another meaning
`str.encode` turns text into bytes. UTF-8 encodes `A` as byte 0x41, the same byte used above for the number 65. `ord` returns a code point; it is not a general text encoder.

In [3]:
letter = "A"
letter_bytes = letter.encode("utf-8")
print("Code point:", f"U+{ord(letter):04X}")
print("UTF-8 hex:", letter_bytes.hex(" "))
print("Same bytes:", letter_bytes == number_bytes)
print("As integer:", int.from_bytes(letter_bytes, "big"))
print("As UTF-8 text:", letter_bytes.decode("utf-8"))
assert letter_bytes == number_bytes

Code point: U+0041
UTF-8 hex: 41
Same bytes: True
As integer: 65
As UTF-8 text: A


---

## 5. A code point can need multiple bytes
The precomposed `é` below is one code point (U+00E9), but two bytes in UTF-8. `len(str)` counts code points, while `len(bytes)` counts bytes. A visible character may also consist of several code points; this example deliberately uses the precomposed form.

In [4]:
text = "\u00e9"  # Precomposed é, not e followed by a combining accent.
encoded = text.encode("utf-8")
print("Text:", text)
print("Code point:", f"U+{ord(text):04X}")
print("Code points:", len(text), "| UTF-8 bytes:", len(encoded))
print("Hex:", encoded.hex(" "))
print("Bits:", " ".join(f"{byte:08b}" for byte in encoded))
assert encoded == bytes([0xC3, 0xA9])
assert len(text) == 1 and len(encoded) == 2

Text: é
Code point: U+00E9
Code points: 1 | UTF-8 bytes: 2
Hex: c3 a9
Bits: 11000011 10101001


---

## 6. Decode with the agreed encoding
The same bytes interpreted as Latin-1 produce different code points without raising an error. Successful decoding alone does not establish that the intended encoding was used.

In [5]:
correct = encoded.decode("utf-8")
wrong = encoded.decode("latin-1")
print("UTF-8:", correct)
print("Latin-1:", wrong)
print("Latin-1 code points:", [f"U+{ord(char):04X}" for char in wrong])
assert correct == text and wrong != text
assert [ord(char) for char in wrong] == [0x00C3, 0x00A9]

UTF-8: é
Latin-1: Ã©
Latin-1 code points: ['U+00C3', 'U+00A9']


---

## 7. Byte order matters for multi-byte integers
Encode the same number in two bytes. Big-endian puts the most significant byte first; little-endian puts it last. Byte order concerns whole bytes, not reversing the bits inside each byte.

In [6]:
number = 258
big = number.to_bytes(2, "big")
little = number.to_bytes(2, "little")
print("Big-endian:", big.hex(" "))
print("Little-endian:", little.hex(" "))
print("Wrong byte order:", int.from_bytes(big, "little"))
assert big == bytes([1, 2]) and little == bytes([2, 1])
assert int.from_bytes(big, "big") == int.from_bytes(little, "little") == number
assert int.from_bytes(big, "little") == 513

Big-endian: 01 02
Little-endian: 02 01
Wrong byte order: 513


---

## Takeaway
Bytes do not contain an automatic type label. Formats and protocols establish how to interpret them. Choose integer width, signedness and byte order explicitly; agree on text encoding. The byte counts here measure serialized data, not Python object memory usage.

---